# Chapter 16: Neural Operators and PDE Equilibria

This solved notebook is the executable counterpart to Chapter 16. It starts
with hand-sized arrays, then carries the same equilibrium contract into
PDE residual correction on a grid.

Goal: a one-dimensional elliptic PDE equilibrium with NumPy grids and PyTorch operators.

Book context:

- Story: Steady-state PDEs are already self-consistency problems: the correct field is the one whose residual vanishes. A DEQ neural operator learns how to settle a function, not just a finite vector.
- State: the discretized function or field $u$
- Source: PDE coefficients, boundary data, and forcing $a,b$
- Operator: a learned residual-correction or Fourier neural operator $G_\theta(a,u)$
- Where it is used: steady-state PDE solvers, operator learning, scientific machine learning, and function-space SILVA analogies

Progression in this notebook:

1. Build a NumPy miniature where every vector, residual, and Jacobian is visible.
2. Rebuild the same map in PyTorch and compare the settled states.
3. Compute derivatives by finite differences, full autograd, VJP/JVP, power iteration, and Hutchinson probes.
4. Derive the mean-field branch from empirical averaging before using its matrix form.
5. Run a tiny SILVA-style layer or the chapter's dataset/project path.
6. Close with the chapter-specific project or recent-paper cell when applicable.

Primary implementation and tutorial references:

- Locus Lab DEQ repo: https://github.com/locuslab/deq
- TorchDEQ repo: https://github.com/locuslab/torchdeq
- Deep Implicit Layers tutorial Colabs: https://implicit-layers-tutorial.org/
- DEQ paper: https://arxiv.org/abs/1909.01377
- MDEQ paper: https://arxiv.org/abs/2006.08656
- Jacobian regularization paper: https://arxiv.org/abs/2106.14342
- Numbered research references: https://jseluis.github.io/silva-networks/paper/references/


In [ ]:

import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "silva_networks").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not find suite root containing src/silva_networks")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt
from silva_networks import (
    np_picard,
    np_finite_difference_jacobian,
    np_exact_tanh_affine_jacobian,
    np_power_iteration,
    np_implicit_gradient,
    SolverConfig,
    picard,
    anderson,
    torch_full_jacobian,
    torch_vjp,
    torch_jvp,
    TinySILVALayer,
    spectral_radius_vjp,
    hutchinson_jacobian_frobenius,
)

np.random.seed(7)
torch.manual_seed(7)
torch.set_default_dtype(torch.float64)


## 1. NumPy miniature: PDE residual correction on a grid


In [ ]:

# NumPy first: a tiny fixed-point layer f(z)=tanh(Wz+s).
n = 4
W_np = 0.35 * np.random.randn(n, n) / np.sqrt(n)
s_np = np.random.randn(n)

def f_np(z):
    return np.tanh(W_np @ z + s_np)

z0_np = np.zeros(n)
trace_np = np_picard(f_np, z0_np, alpha=0.7, max_iter=80, tol=1e-10)
print("NumPy z*:", trace_np.z)
print("NumPy final residual:", trace_np.residuals[-1])

J_exact = np_exact_tanh_affine_jacobian(W_np, trace_np.z, s_np)
J_fd = np_finite_difference_jacobian(f_np, trace_np.z, eps=1e-5)
print("||J_exact - J_fd||_F:", np.linalg.norm(J_exact - J_fd))

T_np = (1 - 0.7) * np.eye(n) + 0.7 * J_exact
rho_np, _ = np_power_iteration(T_np)
print("materialized damped spectral-radius estimate:", rho_np)

plt.plot(trace_np.residuals, marker="o")
plt.yscale("log")
plt.title("NumPy damped Picard residual")
plt.xlabel("iteration")
plt.ylabel("||f(z)-z||")
plt.show()


## 2. PyTorch mirror: PDE residual correction on a grid


In [ ]:

# PyTorch mirror: same W and s, now with autograd.
W = torch.tensor(W_np)
s = torch.tensor(s_np)

def f_torch(z):
    return torch.tanh(W @ z + s)

dim = n
z0 = torch.zeros(n)
trace = picard(
    f_torch,
    z0,
    SolverConfig(alpha=0.7, max_iter=80, tol=1e-10),
)
print("PyTorch z*:", trace.z.detach().numpy())
print("PyTorch final residual:", trace.residuals[-1])
print("NumPy/PyTorch state difference:", np.linalg.norm(trace_np.z - trace.z.detach().numpy()))

J_torch = torch_full_jacobian(f_torch, trace.z)
print("PyTorch full Jacobian shape:", tuple(J_torch.shape))
print("||J_torch - J_exact||_F:", torch.linalg.norm(J_torch - torch.tensor(J_exact)).item())

plt.plot(trace.residuals, marker="o")
plt.yscale("log")
plt.title("PyTorch damped Picard residual")
plt.xlabel("iteration")
plt.ylabel("||f(z)-z||")
plt.show()


## 3. Jacobian products and stability for Chapter 16


In [ ]:

# Scalable Jacobian operations: VJP, JVP, spectral radius, and Hutchinson.
z_star = trace.z.detach()
v = torch.randn_like(z_star)
v = v / torch.linalg.norm(v)

jtv = torch_vjp(f_torch, z_star, v)
_, jv = torch_jvp(f_torch, z_star, v)
print("VJP shape:", tuple(jtv.shape), "JVP shape:", tuple(jv.shape))

rho_torch = spectral_radius_vjp(
    lambda zz: (1 - 0.7) * zz + 0.7 * f_torch(zz),
    z_star,
    iters=50,
)
print("VJP damped spectral-radius estimate:", rho_torch)
print("materialized NumPy estimate:", rho_np)

jac_frob_est = hutchinson_jacobian_frobenius(f_torch, z_star, samples=8)
jac_frob_exact = torch.linalg.norm(J_torch) ** 2
print("Hutchinson estimate of ||J||_F^2:", float(jac_frob_est))
print("Exact ||J||_F^2 for this small case:", float(jac_frob_exact))


## 4. Implicit-gradient adjoint check for Chapter 16


In [ ]:

# Verify the implicit-gradient equation on a scalar parameter.
a = torch.tensor(0.35, requires_grad=True)
b = torch.tensor(1.2)

def scalar_solve(a):
    z = torch.zeros(())
    for _ in range(80):
        z = torch.tanh(a * z + b)
    return z

z_star = scalar_solve(a)
loss = 0.5 * z_star ** 2
loss.backward()

with torch.no_grad():
    jf = a * (1 - torch.tanh(a * z_star + b) ** 2)
    df_da = z_star * (1 - torch.tanh(a * z_star + b) ** 2)
    implicit_grad = z_star * df_da / (1 - jf)
print("autograd through finite solve:", float(a.grad))
print("implicit formula:", float(implicit_grad))

# The same adjoint idea with a materialized NumPy Jacobian.
g_np = trace_np.z.copy()  # gradient of 0.5 ||z*||^2 with respect to z*
df_dscale = (1 - np.tanh(W_np @ trace_np.z + s_np) ** 2)[:, None] * (W_np @ trace_np.z)[:, None]
grad_scale = np_implicit_gradient(J_exact, g_np, df_dscale)
print("NumPy adjoint gradient for scaling W:", grad_scale)


## 5. Mean-field branch for PDE residual correction on a grid

This cell derives the global branch from the empirical average and verifies the compact matrix form before using `MeanFieldGlobal` inside the SILVA layer.


In [ ]:

# From scratch: derive and verify the mean-field global term.
# Start with entity states Y: rows are nodes/atoms/channels, columns are features.
N, d, d_out = 5, 3, 2
Y = np.random.randn(N, d)
Wg = np.random.randn(d_out, d)

# 1. Literal empirical mean from the definition: \bar y = (1/N) sum_j y_j.
g_loop = np.zeros(d)
for j in range(N):
    g_loop = g_loop + Y[j]
g_loop = g_loop / N

# 2. Broadcast the same global context to every entity and mix channels.
out_loop = np.zeros((N, d_out))
for i in range(N):
    out_loop[i] = g_loop @ Wg.T

# 3. Matrix form: G(Y) = (1/N) 1 1^T Y W_g^T.
one = np.ones((N, 1))
P_global = (one @ one.T) / N
out_matrix = P_global @ Y @ Wg.T

print("loop/matrix agreement:", np.linalg.norm(out_loop - out_matrix))
print("rank of node-space mean-field matrix:", np.linalg.matrix_rank(P_global))

# 4. Permutation equivariance: G(Pi Y) = Pi G(Y).
perm = np.array([2, 0, 4, 1, 3])
Pi = np.eye(N)[perm]
lhs = P_global @ (Pi @ Y) @ Wg.T
rhs = Pi @ (P_global @ Y @ Wg.T)
print("permutation equivariance error:", np.linalg.norm(lhs - rhs))

# 5. Batch version: means must be computed inside each graph/sample.
batch = np.array([0, 0, 0, 1, 1])
out_batch = np.zeros((N, d_out))
for b_id in np.unique(batch):
    mask = batch == b_id
    gb = Y[mask].mean(axis=0, keepdims=True)
    out_batch[mask] = gb @ Wg.T
print("batch-local global term shape:", out_batch.shape)


## 6. SILVA smoke test: a one-dimensional elliptic PDE equilibrium with NumPy grids and PyTorch operators


In [ ]:

# Tiny SILVA layer smoke example.
x = torch.randn(8, 3)
edge_index = torch.tensor([[0,1,2,3,4,5,6,7], [1,2,3,4,5,6,7,0]])
layer = TinySILVALayer(in_dim=3, hidden_dim=12, alpha=0.4, max_iter=20)
z, residuals = layer(x, edge_index=edge_index, return_trace=True)
print("state shape:", tuple(z.shape))
print("last residual:", residuals[-1])
rho = spectral_radius_vjp(lambda zz: (1-layer.alpha)*zz + layer.alpha*layer.f(zz, x, edge_index=edge_index), z)
print("damped spectral radius estimate:", rho)


## Chapter-specific recent-paper implementation: Neural Operators and PDE Equilibria


In [ ]:

# Chapter 16: one-dimensional PDE equilibrium.
# Solve -u'' = q on [0, 1] with zero boundary conditions using an equilibrium update.
m = 32
h = 1.0 / (m + 1)
x_grid = np.linspace(h, 1.0 - h, m)
q = np.sin(np.pi * x_grid)
A = (2 * np.eye(m) - np.eye(m, k=1) - np.eye(m, k=-1)) / (h ** 2)

eig_max = np.linalg.eigvalsh(A).max()
alpha = 0.8 / eig_max

def pde_update(u):
    return u + alpha * (q - A @ u)

trace = np_picard(pde_update, np.zeros(m), alpha=1.0, max_iter=500, tol=1e-10)
u_direct = np.linalg.solve(A, q)
print("PDE residual ||Au-q||:", np.linalg.norm(A @ trace.z - q))
print("difference from direct solve:", np.linalg.norm(trace.z - u_direct))

plt.plot(x_grid, trace.z, label="equilibrium")
plt.plot(x_grid, u_direct, "--", label="direct solve")
plt.legend()
plt.title("DEQ-style fixed point for a steady PDE")
plt.show()

# PyTorch mirror: same update, now differentiable with respect to the forcing q.
A_t = torch.tensor(A)
q_t = torch.tensor(q, requires_grad=True)
u = torch.zeros(m)
for _ in range(300):
    u = u + alpha * (q_t - A_t @ u)
loss = 0.5 * torch.sum(u ** 2)
loss.backward()
print("gradient wrt forcing, first five entries:", q_t.grad[:5])


## Data and project path for Chapter 16

For quick experiments, use the companion cache under `data/cache/`.

- Vision: MNIST, Fashion-MNIST, CIFAR-10 small subsets.
- Molecules: ZINC-12k smoke subsets.
- Graphs: Cora/Citeseer/Pubmed public splits and CLUSTER smoke batches.

Downloader:

```bash
python data/download_datasets.py mnist fashion cifar10 zinc planetoid cluster
```

For Chapter 16, the mini-project focus is: **a one-dimensional elliptic PDE equilibrium with NumPy grids and PyTorch operators**.


## Chapter 16 solved notebook summary

The implementation target is **a one-dimensional elliptic PDE equilibrium with NumPy grids and PyTorch operators**. The cells above construct an
equilibrium in NumPy, repeat it in PyTorch, compute Jacobians by several
methods, verify an implicit derivative, and run a tiny SILVA-style local/global
layer. This is the executable baseline for scaling the chapter to the article's
full experiments.


## From 16 Neural Operators Pdes to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | a sampled solution field on a grid or mesh |
| Condition | coefficient, forcing, boundary, and coordinate fields |
| Repeated computation | a tied local/spectral/operator field with source reinjection |
| Required invariants | spatial shape, boundary conditions, and resolution semantics |
| Replaceable components | lifting map, spectral/local operator, physics field, readout, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**solution error, PDE residual, boundary error, and fixed-point residual**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **resolution, retained modes, channels, domain size, and dataset size**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '16_neural_operators_pdes.ipynb',
    "state": 'a sampled solution field on a grid or mesh',
    "condition": 'coefficient, forcing, boundary, and coordinate fields',
    "transition": 'a tied local/spectral/operator field with source reinjection',
    "invariants": 'spatial shape, boundary conditions, and resolution semantics',
    "compact_metric": 'solution error, PDE residual, boundary error, and fixed-point residual',
    "scale_axis": 'resolution, retained modes, channels, domain size, and dataset size',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record
